In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    SpenderID,
    drop_duplicate_columns,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`DSO` and {term}`ET` data is not connected (see [](general:ic)). No joining process is necessary, as the data is in the longitudinal format.

In [ ]:
idcols = ["donor_et_dso", "donor_et_id_et"]
assert len(split_data(data, idcols)) == 2, "Not 2 different row types present!?"

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of tests (see [](general:rf)). We kept all rows.

In [ ]:
data["Institute with a measurement date"] = (
    (~data["sampling_date_dso"].isna()) + (~data["sampling_date_et"].isna()) * 2
).replace({1: "DSO", 2: "ET", 3: "DSO+ET", 0: "No Date"})
data["donor"] = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
data["sampling_date"] = collapse_col(
    data.loc[:, ["sampling_date_dso", "sampling_date_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
display_long_data_doc(
    data,
    [
        "donor",
    ],
    "sampling_date",
    "Institute with a measurement date",
)
data.drop(
    columns=["sampling_date", "donor", "Institute with a measurement date"],
    inplace=True,
)

### Unit Conversions

We applied the common translations (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

### Consolidating Columns

We consolidated columns that appear for {term}`ET` and {term}`DSO` (see [](general:crc))

In [ ]:
red = find_redundant_cols(data)
red.pop("pathogen")
red["donor_et_id_et"] = ["donor_et_dso", "donor_et_id_et"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `sampling_date` column as the time axis.

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["sampling_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
# Another base class might be necessary, see util.py
# describe columns, without checks for now, order is important
pathogens = [
    "Escherichia coli",
    "Staphylococcus aureus",
    "Aspergillus fumigatus",
    "Klebsiella",
    "Sonstige Keime",
    "Enterobacter",
    "Staphylococcus epidermis",
    "Acinetobacter baumanii",
    "Candida albicans",
    "Enterokokken",
    "Streptokokken",
    "Candida glabrata",
    "Gramnegative Stäbchen",
    "Pseudomonas",
    "Pneumokokken",
    "Aspergillus niger",
    "Aerobier",
    "Anaerobier",
    "Chlamydien",
    "Chlamydia psittaci",
]


class DonorPostmortemLabMicrobiology(SpenderID):
    ab_resistant: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Antibiotic resistant",
        description="Was the detected pathogen antibiotic resistant?",
        isin=["negative", "positive", "not tested"],
    )
    communicated_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Communicated date",
        description="Date when the lab result was communicated",
    )
    exam_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Examination type",
        description="How were the pathogens detected?",
        isin=[
            "Erregernachweis",
            "Sonstige Untersuchung",
            "Mikroskopisches Präparat",
            "Pilze",
        ],
    )
    examination_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Examination date",
        description="Date when the lab measurements were conducted",
    )
    pathogen_1: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="First Pathogen",
        description="What was the first pathogen detected?",
        isin=pathogens,
    )
    pathogen_2: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Second Pathogen",
        description="What was the second pathogen detected?",
        isin=pathogens,
    )
    pathogen_3: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Third Pathogen",
        description="What was the third pathogen detected?",
        isin=pathogens,
    )
    pathogen_present: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pathogen detected",
        description="Was a pathogen detected?",
        isin=["kein Erreger nachgewiesen", "Erreger nachgewiesen", "Kontamination"],
    )
    pathogen_present: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pathogen detected",
        description="Was a pathogen detected?",
        isin=["kein Erreger nachgewiesen", "Erreger nachgewiesen", "Kontamination"],
    )
    result_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Result date",
        description="Date when the lab result was generated",
    )
    sampling_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Sampling date",
        description="Date when the sample was taken",
    )
    sampling_tissue: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Tissue",
        description="From what tissue was the sample taken?",
        isin=[
            "Abstrich",
            "Trachealsekret",
            "Bronchiallavage",
            "Katheterurin",
            "Blut",
            "Sonstiges",
            "Liquor",
            "Sammelurin",
            "Punktat",
            "Organ",
            "Drainage",
            "Stuhl",
            "Sputum",
            "Gewebe",
            "Biopsie",
            "Serum",
            "Milz",
        ],
    )
    taking_antibiotics: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Antibiotics",
        description="Was the patient taking antibiotics?",
        isin=["yes", "no"],
    )
    test_comment: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Comment",
        description="Comment on the microbiological test result",
    )
    text_pathogen_count: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pathogen Count",
        description="Description of the pathogen count",
    )

    class Config:
        title = "Donor Postmortem Microbiological Lab Test Dataset"
        description = "Each row represents a microbiological lab test for a deceased donor. The data is based on the 'element_spender_postmortem_labor_mikrobiologie.csv' file. It contains data from the ET and DSO."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemLabMicrobiology, data)

In [ ]:
DonorPostmortemLabMicrobiology.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemLabMicrobiology.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)